# 🧪 Sesión 17 · Control de Acceso Basado en Roles (RBAC)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Qué diferencia hay entre no estar autenticado y no tener permiso?

---

## 🖥️ Así se verá el resultado final

```http
GET /api/admin/panel   Authorization: Bearer <token_admin>
HTTP/1.1 200 OK
{"panel":"admin"}

GET /api/admin/panel   Authorization: Bearer <token_user>
HTTP/1.1 403 Forbidden

GET /api/admin/panel   (sin token)
HTTP/1.1 401 Unauthorized

GET /api/perfil        Authorization: Bearer <token_user>
HTTP/1.1 200 OK
{"usuario":"ana","rol":"user"}
```

_El 403 no es un error de credenciales: es una decisión de autorización._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Incluir el rol dentro de los claims del token.
- Distinguir 401 (no autenticado) de 403 (sin permiso).
- Proteger una ruta administrativa con RBAC.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] El administrador obtiene su token → HTTP 200
- [ ] El usuario normal obtiene su token → HTTP 200
- [ ] El admin entra al panel → HTTP 200
- [ ] El usuario normal recibe 403 → HTTP 403
- [ ] Sin token recibe 401 → HTTP 401
- [ ] El perfil funciona para el usuario normal → HTTP 200

## 🧩 El reto

Implementa autenticación con roles: un usuario normal y un administrador. El panel administrativo solo debe abrirse para admin.

**Requisitos del contrato**

- Usuarios de prueba: admin/admin123 (rol admin) y ana/secreta (rol user).
- El token incluye el rol del usuario en sus claims.
- GET /api/admin/panel responde 200 solo para rol admin.
- Un usuario autenticado con rol user recibe 403.
- Sin token se responde 401, no 403.
- GET /api/perfil funciona para cualquier usuario autenticado.

> **Pista conceptual:** Primero autentica (¿quién eres?) y después autoriza (¿tu rol alcanza?). Si devuelves 403 a quien no tiene token, estás mezclando las dos preguntas.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion17.md` con la matriz rol → permiso.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [4]:
%%writefile app.py
from fastapi import FastAPI, HTTPException, Header
from pydantic import BaseModel
import jwt

app = FastAPI(title="Control de Acceso Basado en Roles")

SECRET_KEY = "mi_clave_super_secreta_para_jwt_2026"

usuarios = {
    "admin": {
        "password": "admin123",
        "rol": "admin"
    },
    "ana": {
        "password": "secreta",
        "rol": "user"
    }
}

class Login(BaseModel):
    usuario: str
    password: str


@app.post("/api/token")
def obtener_token(datos: Login):

    usuario = usuarios.get(datos.usuario)

    if not usuario or usuario["password"] != datos.password:
        raise HTTPException(
            status_code=401,
            detail="Credenciales incorrectas"
        )

    token = jwt.encode(
        {
            "usuario": datos.usuario,
            "rol": usuario["rol"]
        },
        SECRET_KEY,
        algorithm="HS256"
    )

    return {"access_token": token}


def validar_token(authorization):

    if not authorization:
        raise HTTPException(
            status_code=401,
            detail="No autenticado"
        )

    try:
        token = authorization.replace("Bearer ", "")

        datos = jwt.decode(
            token,
            SECRET_KEY,
            algorithms=["HS256"]
        )

        return datos

    except:
        raise HTTPException(
            status_code=401,
            detail="Token inválido"
        )


@app.get("/api/admin/panel")
def panel_admin(Authorization: str = Header(default=None)):

    usuario = validar_token(Authorization)

    if usuario["rol"] != "admin":
        raise HTTPException(
            status_code=403,
            detail="Sin permiso"
        )

    return {"panel": "admin"}


@app.get("/api/perfil")
def perfil(Authorization: str = Header(default=None)):

    usuario = validar_token(Authorization)

    return {
        "usuario": usuario["usuario"],
        "rol": usuario["rol"]
    }


Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Aplicación desarrollada con FastAPI en Python. Se utiliza autenticación mediante tokens JWT y control de acceso basado en roles. El sistema debe diferenciar entre usuarios no autenticados (401) y usuarios sin permisos (403). |
| **Rol** | Actuar como desarrollador backend encargado de implementar el inicio de sesión, la generación de tokens y la restricción de acceso según el rol de cada usuario. |
| **Especificación** | Crear dos usuarios de prueba: admin/admin123 con rol admin y ana/secreta con rol user. El endpoint POST /api/token debe generar un token con el usuario y su rol. El endpoint GET /api/admin/panel solo permite acceso al administrador (200), devuelve 403 para usuarios normales y 401 cuando no se envía token. El endpoint GET /api/perfil debe funcionar para cualquier usuario autenticado y mostrar su usuario y rol. |
| **Output** | Respuestas en formato JSON con los códigos HTTP esperados: 200 para accesos válidos, 401 cuando no hay autenticación y 403 cuando el usuario está autenticado pero no tiene permisos suficientes. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [5]:
from verificar import probar_http

casos = [{'descripcion': 'El administrador obtiene su token',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'admin', 'password': 'admin123'},
  'estado': 200,
  'guardar': {'token_admin': 'access_token'}},
 {'descripcion': 'El usuario normal obtiene su token',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'guardar': {'token_user': 'access_token'}},
 {'descripcion': 'El admin entra al panel',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_admin}'},
  'estado': 200,
  'json_contiene': {'panel': 'admin'}},
 {'descripcion': 'El usuario normal recibe 403',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 403},
 {'descripcion': 'Sin token recibe 401', 'metodo': 'GET', 'ruta': '/api/admin/panel', 'estado': 401},
 {'descripcion': 'El perfil funciona para el usuario normal',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana', 'rol': 'user'}}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 6 casos · app.py
✅ 01. El administrador obtiene su token
✅ 02. El usuario normal obtiene su token
✅ 03. El admin entra al panel
✅ 04. El usuario normal recibe 403
✅ 05. Sin token recibe 401
✅ 06. El perfil funciona para el usuario normal
------------------------------------------------------------------------------
RESULTADO: 6/6 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion17.md` a `auditorias/auditoria_sesion17.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Lo que más me costó fue hacer que el sistema distinguiera entre los errores 401 y 403. Al principio pensaba que eran lo mismo, pero entendí que uno es cuando la persona no ha iniciado sesión y el otro cuando sí inició sesión pero no tiene permiso para entrar.

   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   Decidí guardar los usuarios directamente en un diccionario dentro del código. Así fue más fácil hacer las pruebas sin necesitar una base de datos.

   - ¿Qué error detectó la verificación que tú no habías previsto?
   La verificación me ayudó a notar que una ruta protegida debía responder con 401 cuando no se enviaba ningún token. Yo pensaba que podía responder con 403, pero eso hacía que la prueba fallara.
   
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
